# MNE-Python 加载 EEG 数据集 Demo

演示如何使用 mne.datasets 加载和预览各种内置 EEG 数据集

In [6]:
import mne
print(mne.__version__)
mne.set_config('MNE_DATA', "./datasets")


1.12.0


In [7]:
import os

# 方法一：使用 os.environ（类似字典）
path = os.environ['MNE_DATA']
print(path)

# 方法二：使用 os.getenv()（推荐，可以设置默认值）
path = os.getenv('MNE_DATA')
print(path)

./datasets
./datasets


## 1. 加载 Sample 数据集（MEG + EEG）

In [8]:
print("=" * 60)
print("1. Sample 数据集")
print("=" * 60)



sample_data_path = mne.datasets.sample.data_path()
print("sample_data_path:", sample_data_path)

sample_data_version = mne.datasets.sample.get_version()
print("version:", sample_data_version)


1. Sample 数据集
Fetching 1 file for the sample dataset ...


  0%|                                              | 0.00/1.65G [00:00<?, ?B/s]

Untarring contents of 'd:\AI\claude-code\datasets\MNE-sample-data-processed.tar.gz' to 'd:\AI\claude-code\datasets'


Download complete in 03m16s (1576.2 MB)
sample_data_path: d:\AI\claude-code\datasets\MNE-sample-data
version: 0.12


In [ ]:
raw_fname = sample_data_path.joinpath("MEG", "sample", "sample_audvis_raw.fif")
raw_sample = mne.io.read_raw_fif(raw_fname, preload=True)

print(raw_sample.info)
# 只看 EEG 通道
raw_sample.pick("eeg").plot(duration=5, n_channels=10, title="Sample EEG")

Opening raw data file d:\AI\claude-code\datasets\MNE-sample-data\MEG\sample\sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Reading 0 ... 166799  =      0.000 ...   277.714 secs...
<Info | 21 non-empty values
 acq_pars: ACQch001 110113 ACQch002 110112 ACQch003 110111 ACQch004 110122 ...
 bads: 2 items (MEG 2443, EEG 053)
 ch_names: MEG 0113, MEG 0112, MEG 0111, MEG 0122, MEG 0123, MEG 0121, MEG ...
 chs: 204 Gradiometers, 102 Magnetometers, 9 Stimulus, 60 EEG, 1 EOG
 custom_ref_applied: False
 description: acquisition (megacq) VectorView system at NMR-MGH
 dev_head_t: MEG device -> head transform
 dig: 146 items (3 Cardinal, 4 HPI, 61 EEG, 78 Extra)
 events: 1 item (list)
 experimenter: MEG
 file_id: 4 items (dict)
 highpass: 0.1 Hz
 hpi_meas: 1 item (list)
 hpi_results: 1 item (list)
 lowpass: 172.2 Hz
 meas_d

## 2. 加载 EEGBCI 运动想象数据集

In [ ]:
print("=" * 60)
print("2. EEGBCI 运动想象数据集")
print("=" * 60)

# subject=1, runs: 6=左拳想象, 10=右拳想象, 14=双脚想象
raw_fnames = mne.datasets.eegbci.load_data(subject=1, runs=[6, 10, 14])
raws = [mne.io.read_raw_edf(f, preload=True) for f in raw_fnames]
raw_eegbci = mne.concatenate_raws(raws)

# 设置标准 10-20 电极位置
mne.datasets.eegbci.standardize(raw_eegbci)
montage = mne.channels.make_standard_montage("standard_1005")
raw_eegbci.set_montage(montage)

print(raw_eegbci.info)
raw_eegbci.plot(duration=5, n_channels=10, title="EEGBCI Motor Imagery")

## 3. 加载 ERP CORE 数据集

In [ ]:
print("=" * 60)
print("3. ERP CORE 数据集")
print("=" * 60)

erp_core_path = mne.datasets.erp_core.data_path()
raw_fname = erp_core_path / "ERP-CORE_Subject-001_Task-Flankers_eeg.fif"
raw_erp = mne.io.read_raw_fif(raw_fname, preload=True)

print(raw_erp.info)
raw_erp.plot(duration=5, n_channels=10, title="ERP CORE - Flankers")

## 4. 加载 Sleep PhysioNet 睡眠数据集

In [ ]:
print("=" * 60)
print("4. Sleep PhysioNet 睡眠数据集")
print("=" * 60)

sleep_files = mne.datasets.sleep_physionet.age.fetch_data(subjects=[0], recording=[1])
raw_sleep = mne.io.read_raw_edf(sleep_files[0][0], preload=True)
annot = mne.read_annotations(sleep_files[0][1])
raw_sleep.set_annotations(annot)

print(raw_sleep.info)
print(f"标注类型: {set(raw_sleep.annotations.description)}")
raw_sleep.plot(duration=30, n_channels=4, title="Sleep PhysioNet")

## 5. 综合示例：EEGBCI 从加载到分类的完整流程

In [ ]:
print("=" * 60)
print("5. EEGBCI 完整处理流程")
print("=" * 60)

# 加载数据
raw_fnames = mne.datasets.eegbci.load_data(subject=1, runs=[6, 10, 14])
raw = mne.concatenate_raws([mne.io.read_raw_edf(f, preload=True) for f in raw_fnames])
mne.datasets.eegbci.standardize(raw)
raw.set_montage(mne.channels.make_standard_montage("standard_1005"))

# 滤波 (8-30 Hz, mu + beta 节律)
raw.filter(8.0, 30.0, fir_design="firwin")

In [ ]:
# 提取事件和 Epochs
events, event_id = mne.events_from_annotations(raw)
print(f"事件类型: {event_id}")

epochs = mne.Epochs(
    raw,
    events,
    event_id,
    tmin=-0.5,
    tmax=2.0,
    baseline=None,
    preload=True,
)

print(f"Epochs 数量: {len(epochs)}")
print(f"Epochs 形状: {epochs.get_data().shape}")  # (n_epochs, n_channels, n_times)

In [ ]:
# 可视化
epochs.plot(n_epochs=3, title="EEGBCI Epochs")

In [ ]:
# 平均 ERP
epochs.average().plot(title="EEGBCI 平均 ERP")

In [ ]:
# PSD 功率谱
epochs.compute_psd(fmin=8, fmax=30).plot(title="EEGBCI PSD")

print("\nDemo 完成!")